# WEEK 3 — SQL Analysis

**Dataset:** NYC Airbnb Open Data (2019)  
**Database:** MySQL  
**Table:** `airbnb_listings`

This notebook contains 10 analytical SQL queries and Pandas verification for each query.

## Setup

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

DATA_PATH = r"C:\Users\vinay\airbnb_nyc_2019_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)


Dataset shape: (48895, 16)


In [2]:
# Replace YOUR_MYSQL_PASSWORD with your MySQL password.
# Do not share your password publicly.

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="Vinay@14",
    host="localhost",
    port=3306,
    database="airbnb_analysis"
)

engine = create_engine(connection_url)

with engine.connect():
    print("MySQL connection successful!")


MySQL connection successful!


In [3]:
count_check = pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM airbnb_listings",
    engine
)
count_check


,total_rows
0,48895


## Query 1 — Average Price by Neighbourhood Group

### Question
What is the average Airbnb price for each neighbourhood group?

In [4]:
query1 = """
SELECT neighbourhood_group, ROUND(AVG(price), 2) AS average_price
FROM airbnb_listings
GROUP BY neighbourhood_group
ORDER BY average_price DESC;
"""

result1 = pd.read_sql(query1, engine)
result1

,neighbourhood_group,average_price
0,Manhattan,196.88
1,Brooklyn,124.38
2,Staten Island,114.81
3,Queens,99.52
4,Bronx,87.50


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [5]:
pandas_check1 = (df.groupby("neighbourhood_group", observed=True)["price"].mean()
    .round(2).sort_values(ascending=False).reset_index())
pandas_check1.columns = ["neighbourhood_group", "average_price"]
pandas_check1

,neighbourhood_group,average_price
0,Manhattan,196.88
1,Brooklyn,124.38
2,Staten Island,114.81
3,Queens,99.52
4,Bronx,87.50


## Query 2 — Number of Listings by Room Type

### Question
How many Airbnb listings are available for each room type?

In [6]:
query2 = """
SELECT room_type, COUNT(*) AS listing_count
FROM airbnb_listings
GROUP BY room_type
ORDER BY listing_count DESC;
"""

result2 = pd.read_sql(query2, engine)
result2

,room_type,listing_count
0,Entire home/apt,25409
1,Private room,22326
2,Shared room,1160


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [7]:
pandas_check2 = (df["room_type"].value_counts()
    .rename_axis("room_type").reset_index(name="listing_count"))
pandas_check2

,room_type,listing_count
0,Entire home/apt,25409
1,Private room,22326
2,Shared room,1160


## Query 3 — Average Price by Room Type

### Question
What is the average Airbnb price for each room type?

In [8]:
query3 = """
SELECT room_type, ROUND(AVG(price), 2) AS average_price
FROM airbnb_listings
GROUP BY room_type
ORDER BY average_price DESC;
"""

result3 = pd.read_sql(query3, engine)
result3

,room_type,average_price
0,Entire home/apt,211.79
1,Private room,89.78
2,Shared room,70.13


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [9]:
pandas_check3 = (df.groupby("room_type", observed=True)["price"].mean()
    .round(2).sort_values(ascending=False).reset_index())
pandas_check3.columns = ["room_type", "average_price"]
pandas_check3

,room_type,average_price
0,Entire home/apt,211.79
1,Private room,89.78
2,Shared room,70.13


## Query 4 — Average Availability by Room Type

### Question
How does annual availability differ by room type?

In [10]:
query4 = """
SELECT room_type, ROUND(AVG(availability_365), 2) AS average_availability_days
FROM airbnb_listings
GROUP BY room_type
ORDER BY average_availability_days DESC;
"""

result4 = pd.read_sql(query4, engine)
result4

,room_type,average_availability_days
0,Shared room,162.00
1,Entire home/apt,111.92
2,Private room,111.20


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [11]:
pandas_check4 = (df.groupby("room_type", observed=True)["availability_365"].mean()
    .round(2).sort_values(ascending=False).reset_index())
pandas_check4.columns = ["room_type", "average_availability_days"]
pandas_check4

,room_type,average_availability_days
0,Shared room,162.00
1,Entire home/apt,111.92
2,Private room,111.20


## Query 5 — Top 10 Neighbourhoods by Average Price

### Question
Which neighbourhoods have the highest average price among neighbourhoods with at least 10 listings?

In [12]:
query5 = """
SELECT neighbourhood, COUNT(*) AS listing_count, ROUND(AVG(price), 2) AS average_price
FROM airbnb_listings
GROUP BY neighbourhood
HAVING COUNT(*) >= 10
ORDER BY average_price DESC
LIMIT 10;
"""

result5 = pd.read_sql(query5, engine)
result5

,neighbourhood,listing_count,average_price
0,Tribeca,177,490.64
1,Riverdale,11,442.09
2,Battery Park City,70,367.56
3,Flatiron District,80,341.93
4,Randall Manor,19,336.00
5,NoHo,78,295.72
6,SoHo,358,287.10
7,Midtown,1545,282.72
8,West Village,768,267.68
9,Greenwich Village,392,263.41


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [13]:
pandas_check5 = (df.groupby("neighbourhood", observed=True)["price"]
    .agg(listing_count="count", average_price="mean")
    .query("listing_count >= 10")
    .assign(average_price=lambda x: x["average_price"].round(2))
    .sort_values("average_price", ascending=False).head(10).reset_index())
pandas_check5

,neighbourhood,listing_count,average_price
0,Tribeca,177,490.64
1,Riverdale,11,442.09
2,Battery Park City,70,367.56
3,Flatiron District,80,341.92
4,Randall Manor,19,336.00
5,NoHo,78,295.72
6,SoHo,358,287.10
7,Midtown,1545,282.72
8,West Village,768,267.68
9,Greenwich Village,392,263.41


## Query 6 — Listings Above the Overall Average Price

### Question
How many listings are priced above the overall average Airbnb price?

In [14]:
query6 = """
SELECT COUNT(*) AS listings_above_overall_average
FROM airbnb_listings
WHERE price > (SELECT AVG(price) FROM airbnb_listings);
"""

result6 = pd.read_sql(query6, engine)
result6

,listings_above_overall_average
0,14879


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [15]:
overall_average = df["price"].mean()
pandas_check6 = pd.DataFrame({
    "listings_above_overall_average": [int((df["price"] > overall_average).sum())]
})
pandas_check6

,listings_above_overall_average
0,14879


## Query 7 — Hosts with More Than 10 Listings

### Question
Which hosts have more than 10 listings, and what is their average listing price?

In [16]:
query7 = """
SELECT host_id, COUNT(*) AS listing_count, ROUND(AVG(price), 2) AS average_price
FROM airbnb_listings
GROUP BY host_id
HAVING COUNT(*) > 10
ORDER BY listing_count DESC, average_price DESC;
"""

result7 = pd.read_sql(query7, engine)
result7

,host_id,listing_count,average_price
0,219517861,327,253.20
1,107434423,232,303.15
2,30283594,121,277.53
3,137358866,103,43.83
4,12243051,96,213.03
...,...,...,...
89,10457196,11,152.36
90,4291007,11,92.91
91,164886138,11,86.45
92,229147376,11,48.00


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [26]:
pandas_check7 = (df.groupby("host_id", dropna=False)
    .agg(listing_count=("id", "count"), average_price=("price", "mean"))
    .query("listing_count > 10")
    .assign(average_price=lambda x: x["average_price"].round(2))
    .sort_values(["listing_count", "average_price"], ascending=[False, False])
    .reset_index())
pandas_check7.head(94)

,host_id,listing_count,average_price
0,219517861,327,253.20
1,107434423,232,303.15
2,30283594,121,277.53
3,137358866,103,43.83
4,12243051,96,213.03
...,...,...,...
89,10457196,11,152.36
90,4291007,11,92.91
91,164886138,11,86.45
92,229147376,11,48.00


## Query 8 — Neighbourhood Group Review Activity

### Question
How many listings and reviews are associated with each neighbourhood group?

In [18]:
query8 = """
SELECT neighbourhood_group, COUNT(*) AS listing_count,
       SUM(number_of_reviews) AS total_reviews,
       ROUND(AVG(number_of_reviews), 2) AS average_reviews_per_listing
FROM airbnb_listings
GROUP BY neighbourhood_group
ORDER BY total_reviews DESC;
"""

result8 = pd.read_sql(query8, engine)
result8

,neighbourhood_group,listing_count,total_reviews,average_reviews_per_listing
0,Brooklyn,20104,486574.0,24.20
1,Manhattan,21661,454569.0,20.99
2,Queens,5666,156950.0,27.70
3,Bronx,1091,28371.0,26.00
4,Staten Island,373,11541.0,30.94


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [19]:
pandas_check8 = (df.groupby("neighbourhood_group", observed=True)
    .agg(listing_count=("id", "count"),
         total_reviews=("number_of_reviews", "sum"),
         average_reviews_per_listing=("number_of_reviews", "mean"))
    .assign(average_reviews_per_listing=lambda x: x["average_reviews_per_listing"].round(2))
    .sort_values("total_reviews", ascending=False).reset_index())
pandas_check8

,neighbourhood_group,listing_count,total_reviews,average_reviews_per_listing
0,Brooklyn,20104,486574,24.20
1,Manhattan,21661,454569,20.99
2,Queens,5666,156950,27.70
3,Bronx,1091,28371,26.00
4,Staten Island,373,11541,30.94


## Query 9 — JOIN with Room-Type Summary

### Question
How does each listing's price compare with the average price for its room type?

In [20]:
query9 = """
SELECT a.room_type, a.id, a.price,
       s.average_room_price,
       ROUND(a.price - s.average_room_price, 2) AS difference_from_room_average
FROM airbnb_listings AS a
JOIN (
    SELECT room_type, AVG(price) AS average_room_price
    FROM airbnb_listings
    GROUP BY room_type
) AS s ON a.room_type = s.room_type
ORDER BY difference_from_room_average DESC
LIMIT 10;
"""

result9 = pd.read_sql(query9, engine)
result9

,room_type,id,price,average_room_price,difference_from_room_average
0,Private room,7003697,10000,89.7810,9910.22
1,Private room,9528920,9999,89.7810,9909.22
2,Entire home/apt,13894339,10000,211.7942,9788.21
3,Entire home/apt,22436899,10000,211.7942,9788.21
4,Entire home/apt,4737930,9999,211.7942,9787.21
5,Entire home/apt,31340283,9999,211.7942,9787.21
6,Entire home/apt,23377410,8500,211.7942,8288.21
7,Entire home/apt,2953058,8000,211.7942,7788.21
8,Entire home/apt,22779726,7703,211.7942,7491.21
9,Private room,34895693,7500,89.7810,7410.22


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [21]:
room_summary = (df.groupby("room_type", observed=True)["price"]
    .mean().rename("average_room_price").reset_index())
pandas_check9 = (df[["room_type", "id", "price"]]
    .merge(room_summary, on="room_type", how="inner")
    .assign(difference_from_room_average=lambda x:
            (x["price"] - x["average_room_price"]).round(2))
    .sort_values("difference_from_room_average", ascending=False).head(10))
pandas_check9

,room_type,id,price,average_room_price,difference_from_room_average
9151,Private room,7003697,10000,89.780973,9910.22
12342,Private room,9528920,9999,89.780973,9909.22
29238,Entire home/apt,22436899,10000,211.794246,9788.21
17692,Entire home/apt,13894339,10000,211.794246,9788.21
6530,Entire home/apt,4737930,9999,211.794246,9787.21
40433,Entire home/apt,31340283,9999,211.794246,9787.21
30268,Entire home/apt,23377410,8500,211.794246,8288.21
4377,Entire home/apt,2953058,8000,211.794246,7788.21
29662,Entire home/apt,22779726,7703,211.794246,7491.21
45666,Private room,34895693,7500,89.780973,7410.22


## Query 10 — Neighbourhoods Above the Overall Average

### Question
Which neighbourhoods have an average price higher than the overall Airbnb average?

In [22]:
query10 = """
SELECT neighbourhood, COUNT(*) AS listing_count,
       ROUND(AVG(price), 2) AS average_price
FROM airbnb_listings
GROUP BY neighbourhood
HAVING AVG(price) > (SELECT AVG(price) FROM airbnb_listings)
ORDER BY average_price DESC
LIMIT 10;
"""

result10 = pd.read_sql(query10, engine)
result10

,neighbourhood,listing_count,average_price
0,Fort Wadsworth,1,800.00
1,Woodrow,1,700.00
2,Tribeca,177,490.64
3,Sea Gate,7,487.86
4,Riverdale,11,442.09
5,Prince's Bay,4,409.50
6,Battery Park City,70,367.56
7,Flatiron District,80,341.93
8,Randall Manor,19,336.00
9,NoHo,78,295.72


### Pandas Verification

The SQL result is verified against the equivalent Pandas calculation.

In [23]:
overall_average = df["price"].mean()
pandas_check10 = (df.groupby("neighbourhood", observed=True)
    .agg(listing_count=("id", "count"), average_price=("price", "mean"))
    .query("average_price > @overall_average")
    .assign(average_price=lambda x: x["average_price"].round(2))
    .sort_values("average_price", ascending=False).head(10).reset_index())
pandas_check10

,neighbourhood,listing_count,average_price
0,Fort Wadsworth,1,800.00
1,Woodrow,1,700.00
2,Tribeca,177,490.64
3,Sea Gate,7,487.86
4,Riverdale,11,442.09
5,Prince's Bay,4,409.50
6,Battery Park City,70,367.56
7,Flatiron District,80,341.92
8,Randall Manor,19,336.00
9,NoHo,78,295.72


## Export Sample Output

In [24]:
sample_parts = []
for i, result in enumerate(
    [result1, result2, result3, result4, result5,
     result6, result7, result8, result9, result10], 1
):
    temp = result.head(10).copy()
    temp.insert(0, "query_number", i)
    sample_parts.append(temp)

sql_sample_output = pd.concat(sample_parts, ignore_index=True, sort=False)
sql_sample_output.to_csv("SQL_Sample_Output.csv", index=False)

print("Saved: SQL_Sample_Output.csv")


Saved: SQL_Sample_Output.csv
